# HOMEWORK 12

In this homework you are going to inspect the GTSDB (German Traffic Sign Detection Benchmark) dataset. The dataset contains images of various classes of traffic signs used in Germany (and the whole EU). The objective of this homework is to go through the steps described below and to implement the necessary code.

At the end, as usual, there will be a couple of questions for you to answer. In addition, the last section of this homework is optional and, if you chose to do it, you'll earn extra point :-)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

### Step 0

Go to the GTSRB dataset official site ([link](https://benchmark.ini.rub.de/gtsrb_dataset.html)) to learn more about the dataset.

### Step 1

Download the dataset ([link](https://www.kaggle.com/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign)) and unzip it.

### Step 2

For this homework, you will be working with the training set. Check out the `Train.csv`, open it and see what it contains. Load the dataset and plot random samples.

In [ ]:
# загружаю данные
root = './GTSRB'  # путь к датасету
data = pd.read_csv(os.path.join(root, 'Train.csv'))

num_samples = len(data)

for ii in range(15):
    idx = np.random.randint(0, num_samples)
    img = cv2.imread(os.path.join(root, data.iloc[idx]['Path']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(3,5,ii+1), plt.imshow(img), plt.title(data.iloc[idx]['ClassId'])

### Step 3

Inspect the dataset by computing and plotting the per-class histogram.

In [ ]:
ids = data['ClassId']

Compute the per class histogram. You can use any approach you want (e.g. `numpy`). It's also worth looking at the `Counter` function from the `collections` module ([link](https://docs.python.org/3/library/collections.html#collections.Counter)) ;-)

In [ ]:
from collections import Counter
hist = Counter(ids)

plt.bar(hist.keys(), hist.values()), plt.grid(True)
plt.xlabel('Traffic Sign ID'), plt.ylabel('Counts')

### Вопросы

* Do you consider the dataset to be balanced? If so, why? If not, why?
* Are there any classes that are (significantly) over-represented or under-represeneted?

**Мои ответы:**

* Нет, датасет не сбалансированный. На гистограмме видно, что столбики очень разной высоты, у одних классов изображений намного больше, чем у других. Если бы датасет был сбалансированным, у всех классов было бы примерно одинаковое количество картинок.

* Да, некоторые классы явно перепредставлены, например классы 1 и 2 (знаки ограничения скорости), у них очень много картинок, наверное потому что эти знаки часто встречаются на дорогах. А некоторые классы наоборот имеют совсем мало примеров (например класс 0 или 19). Это может быть проблемой при обучении модели, она будет лучше распознавать частые классы и хуже редкие.

### Optional

Perform a further analysis on the dataset and draw some conclusion from it.

Hint 1: Unlike MNIST or CIFAR10, this dataset contains images with various spatial resolutions. Is there anything we can tell about the resolution distribution?
Hint 2: What about the brightness distribution? Are there classes there are significantly more bright than others?

In [ ]:
# анализ разрешений
widths = data['Width'].values
heights = data['Height'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(widths, bins=50, color='salmon', edgecolor='black')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Image Widths')
axes[0].grid(True)

axes[1].hist(heights, bins=50, color='skyblue', edgecolor='black')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Image Heights')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Ширина: от {widths.min()} до {widths.max()} пикселей")
print(f"Высота: от {heights.min()} до {heights.max()} пикселей")
print(f"Средний размер: {widths.mean():.0f} x {heights.mean():.0f}")

In [ ]:
# анализ яркости по классам
from collections import defaultdict

brightness_per_class = defaultdict(list)
sample_per_class = 50

for class_id in sorted(data['ClassId'].unique()):
    class_data = data[data['ClassId'] == class_id]
    if len(class_data) > sample_per_class:
        class_data = class_data.sample(sample_per_class, random_state=42)
    
    for _, row in class_data.iterrows():
        img = cv2.imread(os.path.join(root, row['Path']))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            brightness_per_class[class_id].append(np.mean(gray))

avg_brightness = {k: np.mean(v) for k, v in brightness_per_class.items()}

plt.figure(figsize=(15, 5))
plt.bar(avg_brightness.keys(), avg_brightness.values(), color='gold', edgecolor='black')
plt.xlabel('Traffic Sign Class ID')
plt.ylabel('Average Brightness')
plt.title('Average Brightness per Class')
plt.grid(True)
plt.show()

brightest = max(avg_brightness, key=avg_brightness.get)
darkest = min(avg_brightness, key=avg_brightness.get)
print(f"Самый яркий класс: {brightest} (яркость = {avg_brightness[brightest]:.1f})")
print(f"Самый тёмный класс: {darkest} (яркость = {avg_brightness[darkest]:.1f})")

**Мои выводы:**

Картинки в датасете очень разного размера, в отличие от MNIST или CIFAR10. Большинство довольно маленькие (около 30-50 пикселей), но есть и крупные. Перед обучением модели нужно будет привести все к одному размеру.

По яркости тоже есть различия между классами. Это логично, у знаков разные цвета и фон. Предупреждающие знаки (жёлтые) ярче, а запрещающие (тёмно-красные/синие) темнее. Возможно стоит нормализовать яркость или использовать аугментацию, чтобы модель не полагалась на яркость вместо формы знака.